In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

#dir = "/cluster/projects/bhklab/Users/julian/PMLB"
dir = "/Users/julianguyen/Documents/multimodal-pmlb"
os.chdir(dir)
sys.path.insert(0, dir)
from workflow.notebooks.utils.early_helpers import *

np.random.seed(101)

In [ ]:
# load in files
atc = pd.read_csv("data/procdata/model_cg1/atc.csv", index_col=0)
rna = pd.read_csv("data/procdata/model_cg1/rna.csv", index_col=0)
cnv = pd.read_csv("data/procdata/model_cg1/cnv.csv", index_col=0)
mut = pd.read_csv("data/procdata/model_cg1/mut.csv", index_col=0)
pheno = pd.read_csv("data/procdata/model_cg1/meta.csv", index_col=0)

# get targest
dt = pheno["doubling_rate"]

# log normalize RNA counts
rna = np.log2(rna + 1)

# scale CNV to -1:1 (currently -2:2)
cnv = cnv/2

# binarize mutations
mut = (mut > 0).astype(int)

# Feature Selection via LASSO Selection Distribution

In [ ]:
run_lasso(atc, dt, "4-CommonGene1/lasso/atac_", repeats=50, features=True)
run_lasso(rna, dt, "4-CommonGene1/lasso/rna_", repeats=50, features=True)
run_lasso(cnv, dt, "4-CommonGene1/lasso/cnv_", repeats=50, features=True)
run_lasso(mut, dt, "4-CommonGene1/lasso/mut_", repeats=50, features=True)

In [ ]:
al = pd.read_csv("data/results/data/4-CommonGene1/lasso/atac_lasso_feature_stability.csv", index_col=0)
rl = pd.read_csv("data/results/data/4-CommonGene1/lasso/rna_lasso_feature_stability.csv", index_col=0)
cl = pd.read_csv("data/results/data/4-CommonGene1/lasso/cnv_lasso_feature_stability.csv", index_col=0)
ml = pd.read_csv("data/results/data/4-CommonGene1/lasso/mut_lasso_feature_stability.csv", index_col=0)

al.head()

In [ ]:
def plot_features_cumulative(df, title, ax):

    bins = np.arange(0, 1 + 0.1, 0.1)

    counts, bins, patches = ax.hist(
        df["frequency"],
        bins=bins,
        cumulative=-1,
        edgecolor="black",
        color="#998888",
        alpha=0.85
    )

    for count, left, right in zip(counts, bins[:-1], bins[1:]):
        if count > 0:
            x = (left + right) / 2
            ax.text(x, count + 0.3, f"{int(count)}",
                    ha="center", va="bottom", fontsize=9)

    ax.set_title(title, fontsize=11, weight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    ax.grid(False, axis="x")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(9, 7))

plot_features_cumulative(al, "ATAC", ax=axes[0, 0])
plot_features_cumulative(rl, "RNA",  ax=axes[0, 1])
plot_features_cumulative(cl, "CNV",  ax=axes[1, 0])
plot_features_cumulative(ml, "MUT",  ax=axes[1, 1])

for ax in axes.flat:
    ax.invert_xaxis()
    ax.set_xlim(1, 0)
    ax.set_xticks(np.arange(0, 1.1, 0.1))
    ax.tick_params(axis='x', which='major', length=4)

fig.text(0.5, 0.04, "Frequency threshold", ha="center", fontsize=12)
fig.text(0.04, 0.5, "Number of genes ≥ threshold", va="center", rotation="vertical", fontsize=12)

plt.tight_layout(rect=[0.05, 0.05, 1, 1])
plt.show()

In [ ]:
a_stable = al[al['frequency'] >= 0.3]
print(a_stable.shape[0])

r_stable = rl[rl['frequency'] >= 0.3]
print(r_stable.shape[0])

c_stable = cl[cl['frequency'] >= 0.3]
print(c_stable.shape[0])

m_stable = ml[ml['frequency'] >= 0.3]
print(m_stable.shape[0])

In [ ]:
# keep features that are selected by at least two modalities
dfs = [a_stable, r_stable, c_stable, m_stable]
row_counts = Counter()
for df in dfs:
    row_counts.update(df.index)
common_features = {row for row, count in row_counts.items() if count >= 2}
len(common_features)

In [ ]:
# subset omics matrices for selected features
omics = [atc, rna, cnv, mut]
selected_features = [
    df.loc[:, df.columns.intersection(common_features)]
    for df in omics
]
atc_selected, rna_selected, cnv_selected, mut_selected = selected_features

print(atc_selected.shape)
print(rna_selected.shape)
print(cnv_selected.shape)
print(mut_selected.shape)

# Unimodal without Cancer Type

In [ ]:
# subset each omics for omic-specific stable features
atc_uni_selected = atc.loc[:, atc.columns.intersection(a_stable.index)]
rna_uni_selected = rna.loc[:, rna.columns.intersection(r_stable.index)]
cnv_uni_selected = cnv.loc[:, cnv.columns.intersection(c_stable.index)]
mut_uni_selected = mut.loc[:, atc.columns.intersection(m_stable.index)]

print(atc_uni_selected.shape)
print(rna_uni_selected.shape)
print(cnv_uni_selected.shape)
print(mut_uni_selected.shape)

### Unimodal Models with 10 Repeats with Unimodal Selected Features

In [ ]:
# ATAC
assert (atc_uni_selected.index == dt.index).all(), "sample mismatch"
a_en = run_elastic_net(atc_uni_selected, dt, "4-CommonGene1/unimodal/atac_", repeats=10, preds=True, features=True)
a_rf = run_random_forest(atc_uni_selected, dt, "4-CommonGene1/unimodal/atac_", repeats=10, preds=True, features=True)

# RNA
assert (rna_uni_selected.index == dt.index).all(), "sample mismatch"
r_en = run_elastic_net(rna_uni_selected, dt, "4-CommonGene1/unimodal/rna_", repeats=10, preds=True, features=True)
r_rf = run_random_forest(rna_uni_selected, dt, "4-CommonGene1/unimodal/rna_", repeats=10, preds=True, features=True)

# CNV
assert (cnv_uni_selected.index == dt.index).all(), "sample mismatch"
c_en = run_elastic_net(cnv_uni_selected, dt, "4-CommonGene1/unimodal/cnv_", repeats=10, preds=True, features=True)
c_rf = run_random_forest(cnv_uni_selected, dt, "4-CommonGene1/unimodal/cnv_", repeats=10, preds=True, features=True)


# MUT
assert (mut_uni_selected.index == dt.index).all(), "sample mismatch"
m_en = run_elastic_net(mut_uni_selected, dt, "4-CommonGene1/unimodal/mut_", repeats=10, preds=True, features=True)
m_rf = run_random_forest(mut_uni_selected, dt, "4-CommonGene1/unimodal/mut_", repeats=10, preds=True, features=True)

### EF Model with Unimodal Selected Features

### LF Model with Unimodal Selected Features

# Early Fusion without Cancer Type

In [ ]:
# TODO:: add unique colnames for each modality
# TODO:: keep only modality where feature is selected?

# scale ATAC, RNA, and CNV
continuous_df = pd.concat([atc_selected, rna_selected, cnv_selected], axis=1)
scaler = StandardScaler()
continuous_scaled = pd.DataFrame(scaler.fit_transform(continuous_df),
                                 index=continuous_df.index,
                                 columns=continuous_df.columns)

# combine dataframes for early fusion
X = pd.concat([continuous_scaled, mut_selected], axis=1)
assert (X.index == dt.index).all(), "sample mismatch"

In [ ]:
run_elastic_net(X, dt, "4-CommonGene1/ef_all")
run_random_forest(X, dt, "4-CommonGene1/ef_all")

# Late Fusion without Cancer Type

In [ ]:
# TODO: run models on _selected matrices, return preds
# TODO: try different weights for late fusion